# Basics of Mobile Robotics - Project Report

Here is the file for the report of our final project uvguz

## Filtering
### Introduction

To ensure robust position estimation of the Thymio robot in the environment, we implemented an **Extended Kalman Filter (EKF)** that fuses data from two sources:
- Visual measurements from the camera (position x, y and orientation θ)
- Motor commands and odometry (left and right wheel speeds)

The filter compensates for measurement uncertainties and ensures continuous position estimation even when the camera temporarily loses sight of the robot.

### State Model

The filter's state vector is defined as:

$$\mathbf{x} = \begin{bmatrix} x \\ y \\ \theta \\ v_L \\ v_R \end{bmatrix}$$

where:
- $x, y$ : robot position in mm
- $\theta$ : robot orientation in radians (positive counterclockwise)
- $v_L, v_R$ : left and right wheel speeds in mm/s

### Kinematic Model

The state transition matrix **A** is computed from the robot's differential kinematic model:

$$\mathbf{A} = \begin{bmatrix}
1 & 0 & -\sin(\theta)\Delta t\frac{v_L-v_R}{2} & \frac{\Delta t}{2}\cos(\theta) & \frac{\Delta t}{2}\cos(\theta) \\
0 & 1 & \cos(\theta)\Delta t\frac{v_L-v_R}{2} & \frac{\Delta t}{2}\sin(\theta) & \frac{\Delta t}{2}\sin(\theta) \\
0 & 0 & 1 & -\frac{\Delta t}{L} & \frac{\Delta t}{L} \\
0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 1
\end{bmatrix}$$

where:
- $\Delta t$ : time step between two measurements
- $L = 122.7$ mm : robot wheelbase

This matrix models the state evolution as a function of wheel speeds and elapsed time.

### Covariance Matrices
#### Matrix Q - Process Noise
The **Q** matrix represents the uncertainty in the motion model:

$$\mathbf{Q} = \begin{bmatrix}
0.17 & 0 & 0 & 0 & 0 \\
0 & 0.17 & 0 & 0 & 0 \\
0 & 0 & 0.05 & 0 & 0 \\
0 & 0 & 0 & 0.43 & 0 \\
0 & 0 & 0 & 0 & 0.45
\end{bmatrix}$$

The values reflect:
- Position uncertainty: 0.17 mm² in x and y
- Orientation uncertainty: 0.05 rad²
- Speed uncertainty: 0.43 and 0.45 (mm/s)²

#### Matrix R - Measurement Noise
The **R** matrix adapts according to camera availability:

**With camera**:
$$\mathbf{R}_{camera} = \begin{bmatrix}
Rx & 0 & 0 & 0 & 0 \\
0 & Ry & 0 & 0 & 0 \\
0 & 0 & R\theta & 0 & 0 \\
0 & 0 & 0 & Rvl & 0 \\
0 & 0 & 0 & 0 & Rvr
\end{bmatrix}$$

The values $Rx$, $Ry$, and $R\theta$ represent the uncertainties in position and orientation from the camera, while $Rvl$ and $Rvr$ correspond to wheel speed measurement uncertainties. These values are determined based on sensor characteristics and empirical tuning.


**Without camera**:
$$\mathbf{R}_{no\ camera} = \begin{bmatrix}
\infty & 0 & 0 & 0 & 0 \\
0 & \infty & 0 & 0 & 0 \\
0 & 0 & \infty & 0 & 0 \\
0 & 0 & 0 & Rvl & 0 \\
0 & 0 & 0 & 0 & Rvr
\end{bmatrix}$$
When the camera is unavailable, position uncertainties are set to infinity, forcing the filter to ignore these measurements and rely only on odometry.

### Kalman Filter Algorithm

The filter follows the classical Kalman filter steps at each iteration:

#### 1. Prediction
$$\hat{\mathbf{x}}_{k|k-1} = \mathbf{A} \hat{\mathbf{x}}_{k-1|k-1}$$
$$\mathbf{P}_{k|k-1} = \mathbf{A} \mathbf{P}_{k-1|k-1} \mathbf{A}^T + \mathbf{Q}$$

#### 2. Innovation
$$\mathbf{y}_k = \mathbf{z}_k - \mathbf{C} \hat{\mathbf{x}}_{k|k-1}$$
$$\mathbf{S}_k = \mathbf{C} \mathbf{P}_{k|k-1} \mathbf{C}^T + \mathbf{R}$$

where $\mathbf{C} = \mathbf{I}_5$ (identity observation matrix since all state components are measured).

#### 3. Kalman Gain
$$\mathbf{K}_k = \mathbf{P}_{k|k-1} \mathbf{C}^T \mathbf{S}_k^{-1}$$

#### 4. Correction
$$\hat{\mathbf{x}}_{k|k} = \hat{\mathbf{x}}_{k|k-1} + \mathbf{K}_k \mathbf{y}_k$$
$$\mathbf{P}_{k|k} = (\mathbf{I} - \mathbf{K}_k \mathbf{C}) \mathbf{P}_{k|k-1}$$

### Implementation

The `Filter` class encapsulates the Kalman filter implementation. It is initialized with the initial position measured by the camera (with an initial variance of 0.01):

```python
filter = Filter(thymio)  # thymio contains [x, y, theta]
```

At each iteration, the `update()` method is called with:
- `measured_pos` : position measured by the camera (or `None` if unavailable)
- `Ts` : time elapsed since the last update
- `ls, rs` : measured left and right wheel speeds

The filter returns the estimated position `[x, y, theta]` which is used for robot navigation.

### Advantages and Robustness

The Kalman filter implementation provides several essential advantages:

1. **Sensor fusion** : Optimally combines precise visual measurements from the camera with continuous motor odometry.

2. **Vision loss tolerance** : When the camera loses the robot (occlusion, out of field of view), the filter continues to estimate position using only odometry by automatically adapting the R matrix.

3. **Noise reduction** : Noisy measurements from the camera and encoders are filtered to produce a smooth and consistent estimate.

4. **Adapted calibration** : Covariance parameters were empirically tuned to reflect the system's actual uncertainties:
   - Very precise camera measurements (0.25 mm² in position)
   - Higher odometry uncertainties (3.46-5.05 (mm/s)² on speeds)

5. **Real-time performance** : The algorithm is lightweight enough to run at each control loop iteration without slowing down the system.

This approach ensures robust and accurate robot localization, essential for autonomous navigation and trajectory tracking.